# C.3 v5 — D-sweep diagnostic + larger-n via historical pooling

**Why this notebook.** v4 robustness probe walked back v3's graduation claim: same operating point with seeds 10..19 gave Δ ≈ +0.03 instead of +0.08 (CI-overlapping), and D=2048 reversed direction. Two open questions remain:

1. **What is the true effect size at the original operating point?** Pool more seeds; v3 (seeds 0..9) + v4 replicate (seeds 10..19) + v5 D=4096 new (seeds 30..39) = n=30 if all aggregated.
2. **What does the D-dependence look like?** Is D=2048's reversal monotonic, non-monotonic, or some other shape? Sweep D ∈ {1024, 2048, 4096, 8192}.

**4 D values × 10 fresh seeds (30..39) = 40 parallel CUDA subprocesses.**

| probe | β | D | seeds | role |
|---|---:|---:|---|---|
| `Dsweep_D1024` | 10 | 1024 | 30..39 | small-D probe |
| `Dsweep_D2048` | 10 | 2048 | 30..39 | confirms v4 reversal with fresh seeds |
| `Dsweep_D4096` | 10 | 4096 | 30..39 | reference; pools with v3 + v4 replicate → n=30 |
| `Dsweep_D8192` | 10 | 8192 | 30..39 | large-D probe |

All else fixed at the v3 graduation config: `wikitext, lr_pull=0.1, n_events=1000, alpha_anti=0.01, repulsion_step_size=0.05, window=8, vocab_cap=1000`.

**Aggregation cell does two things:**
1. D-sweep table: per-D Δ + CI at n=10 (new seeds only) — characterizes D-dependence shape.
2. Pooled larger-n estimate: reads v3 + v4 replicate JSONs from Drive, pools with v5 D=4096, reports Wilson CI on n≈30 successes/trials at the original op point. Falls back gracefully if Drive data is missing.

**Decision tree after this run:**
- If D-sweep is non-monotonic AND pooled n=30 is CI-disjoint at D=4096: real but D-specific effect; write Report 112 as an operating-point sensitivity study and recommend Path γ.
- If D=2048 reversal reproduces with new seeds AND pooled n=30 is CI-overlapping: signal is too weak/fragile to graduate; clean Path γ pivot.
- If D-sweep is monotonic increasing in D AND pooled n=30 is CI-disjoint: substrate-saturation story for the mechanism; write report claiming graduation only at D ≥ 4096 with appropriate caveats.

In [ ]:
# 1. Clone the repo and apply three patches.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout codex/phase5-prime-bundle-first-scene-memory
!git log --oneline -3

import subprocess
pre_pull = subprocess.run(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py'], capture_output=True, text=True).stdout.strip()
pre_gram = subprocess.run(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py'], capture_output=True, text=True).stdout.strip()
pre_wt   = subprocess.run(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py'], capture_output=True, text=True).stdout.strip()
print(f'pre-patch tracers: --lr-pull = {pre_pull}; kernel-trick gram = {pre_gram}; Salesforce/wikitext = {pre_wt}')

patch = r'''diff --git a/experiments/c3_phase3_exit_criterion.py b/experiments/c3_phase3_exit_criterion.py
--- a/experiments/c3_phase3_exit_criterion.py
+++ b/experiments/c3_phase3_exit_criterion.py
@@ -576,6 +576,8 @@ def _run_single_seed_condition(
     k: int,
     alpha_anti: float,
     repulsion_step_size: float,
+    lr_pull: float,
+    lr_push: float,
     device: str,
     repo_root: Path,
     wikitext_corpus: Optional[_WikiTextCorpus] = None,
@@ -756,6 +758,8 @@ def _run_single_seed_condition(
             vocab_size=vocab_size,
             n_events=n_consolidation_events,
             device=device,
+            lr_pull=lr_pull,
+            lr_push=lr_push,
             repulsion_step_size=repulsion_step_size,
         )
 
@@ -838,6 +842,8 @@ def run(
     n_consolidation_events: int = 1000,
     alpha_anti: float = 0.0,
     repulsion_step_size: float = 0.0,
+    lr_pull: float = 0.1,
+    lr_push: float = 0.05,
     device: str,
     output_dir: Path,
     repo_root: Path,
@@ -919,6 +925,8 @@ def run(
                     k=k,
                     alpha_anti=alpha_anti,
                     repulsion_step_size=repulsion_step_size,
+                    lr_pull=lr_pull,
+                    lr_push=lr_push,
                     device=device,
                     repo_root=repo_root,
                     wikitext_corpus=wikitext_corpus,
@@ -1010,6 +1018,8 @@ def run(
             "substrate_repulsion_active": bool(
                 alpha_anti > 0.0 and repulsion_step_size > 0.0
             ),
+            "lr_pull": float(lr_pull),
+            "lr_push": float(lr_push),
             "operating_point": {
                 "D": D,
                 "landscape_size": landscape_size,
@@ -1370,6 +1380,27 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
             "smoke (no inter-atom-separability force)."
         ),
     )
+    parser.add_argument(
+        "--lr-pull",
+        type=float,
+        default=0.1,
+        help=(
+            "Per-event consolidation pull learning rate (OnlineCodebookUpdater "
+            "lr_pull). Default 0.1 matches the existing Path α smoke. Sweep "
+            "above this to test whether consolidation strength is too weak "
+            "to express corpus-specific learning at the synthetic operating "
+            "point."
+        ),
+    )
+    parser.add_argument(
+        "--lr-push",
+        type=float,
+        default=0.05,
+        help=(
+            "Per-event consolidation push learning rate (OnlineCodebookUpdater "
+            "lr_push). Default 0.05 matches the existing Path α smoke."
+        ),
+    )
     parser.add_argument(
         "--repulsion-step-size",
         type=float,
@@ -1468,6 +1499,8 @@ def main(argv: Optional[Sequence[str]] = None) -> int:
         n_consolidation_events=args.n_consolidation_events,
         alpha_anti=args.alpha_anti,
         repulsion_step_size=args.repulsion_step_size,
+        lr_pull=args.lr_pull,
+        lr_push=args.lr_push,
         device=args.device,
         output_dir=output_dir,
         repo_root=repo_root,
diff --git a/src/energy_memory/phase4/consolidation.py b/src/energy_memory/phase4/consolidation.py
--- a/src/energy_memory/phase4/consolidation.py
+++ b/src/energy_memory/phase4/consolidation.py
@@ -639,10 +639,29 @@ class ConsolidationState:
         # Hermitian Gram of centered basin members. For complex (FHRR)
         # tensors, diffs.conj().T @ diffs is Hermitian → real eigenvalues
         # via torch.linalg.eigh.
-        sigma = (diffs.conj().transpose(-1, -2) @ diffs) / float(n)
+        # Compute the eigenvalues of σ = diffs.conj().T @ diffs / n via the
+        # n×n Gram matrix gram = diffs @ diffs.conj().T / n instead of the
+        # D×D scatter matrix. The two matrices share exactly the same set
+        # of non-zero eigenvalues (standard "kernel trick" identity); the
+        # D×D form additionally carries (D - n) trivial zero eigenvalues
+        # because rank(σ) ≤ n_members ≤ basin_trace_buffer_size (64) ≪ D
+        # (4096 by default in this project). That (D - n) zero subspace
+        # makes σ numerically ill-conditioned at the precision available
+        # to torch.linalg.eigvalsh — observed on Colab CUDA at 2026-05-27
+        # as LinAlgError 4095 and even on CPU LAPACK as LinAlgError 5/12.
+        # The n×n Gram path is full-rank for non-degenerate samples and
+        # an order of magnitude smaller (4 KB vs 16 MB at D=4096, n=8).
+        # Mathematically byte-identical at the λ_1 / λ_2 layer used below;
+        # the C.2.2 dynamic's behavior is unchanged.
+        gram = (diffs @ diffs.conj().transpose(-1, -2)) / float(n)
         # Eigh returns ascending eigenvalues. Take top two: λ_1 (last),
         # λ_2 (second-to-last). All ops stay on-device.
-        eigvals = torch.linalg.eigvalsh(sigma)
+        try:
+            eigvals = torch.linalg.eigvalsh(gram)
+        except torch._C._LinAlgError:
+            # Defensive: keep the CPU fallback in case some pathological
+            # input still trips cuSOLVER (e.g. identical basin members).
+            eigvals = torch.linalg.eigvalsh(gram.cpu()).to(gram.device)
         lam_1 = eigvals[-1]
         lam_2 = eigvals[-2] if eigvals.shape[0] >= 2 else torch.zeros_like(lam_1)
         # Clamp at 0 — eigh may return tiny negatives for near-singular Σ.
diff --git a/src/energy_memory/phase2/corpus.py b/src/energy_memory/phase2/corpus.py
--- a/src/energy_memory/phase2/corpus.py
+++ b/src/energy_memory/phase2/corpus.py
@@ -113,7 +113,13 @@ def load_repo_sample_splits(repo_root: Path) -> Dict[str, List[str]]:
 def load_wikitext_splits(name: str = "wikitext-2-raw-v1") -> Dict[str, List[str]]:
     if load_dataset is None:  # pragma: no cover - exercised only when dependency missing
         raise ModuleNotFoundError("datasets is required to load WikiText-2")
-    dataset = load_dataset("wikitext", name)
+    # Use the canonical Salesforce/wikitext namespace. The bare "wikitext"
+    # form worked with older HF stacks but recent huggingface_hub versions
+    # (~0.30+) ship a stricter HF URI parser that rejects any repo id
+    # without an explicit namespace, raising HfUriError. The Salesforce
+    # mirror is the current canonical home of the dataset; config names
+    # ("wikitext-2-raw-v1", "wikitext-103-raw-v1", ...) are unchanged.
+    dataset = load_dataset("Salesforce/wikitext", name)
     return {
         "train": [row["text"] for row in dataset["train"]],
         "validation": [row["text"] for row in dataset["validation"]],
'''

with open('/tmp/c3_combined.patch', 'w') as f:
    f.write(patch)
check = subprocess.run(['git', 'apply', '--check', '/tmp/c3_combined.patch'], capture_output=True, text=True)
if check.returncode == 0:
    subprocess.check_call(['git', 'apply', '/tmp/c3_combined.patch'])
    print('combined patch applied.')
else:
    if int(pre_pull or '0') >= 1 and int(pre_gram or '0') >= 1 and int(pre_wt or '0') >= 1:
        print('all three patches already in branch — skipping apply.')
    else:
        print('PATCH APPLY FAILED:'); print(check.stderr)
        raise SystemExit('Cannot continue.')

post_pull = subprocess.check_output(['grep', '-c', '--', '--lr-pull', 'experiments/c3_phase3_exit_criterion.py']).decode().strip()
post_gram = subprocess.check_output(['grep', '-c', 'kernel trick', 'src/energy_memory/phase4/consolidation.py']).decode().strip()
post_wt   = subprocess.check_output(['grep', '-c', 'Salesforce/wikitext', 'src/energy_memory/phase2/corpus.py']).decode().strip()
print(f'post-patch tracers: --lr-pull = {post_pull}; kernel-trick = {post_gram}; Salesforce/wikitext = {post_wt}')
assert int(post_pull) >= 1 and int(post_gram) >= 1 and int(post_wt) >= 1, 'patches missing'

In [ ]:
# 2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuro-ai/results', exist_ok=True)
# Check whether historical v3 + v4 replicate JSONs are present so we know
# whether the pooled n≈30 aggregation will work.
from pathlib import Path
v3_dir = Path('/content/drive/MyDrive/neuro-ai/results/c3_followup_v3_2026-05-27')
v4_dir = Path('/content/drive/MyDrive/neuro-ai/results/c3_robustness_v4_2026-05-27')
v3_n10_dirs = list(v3_dir.glob('n10_wikitext_base_seed*')) if v3_dir.exists() else []
v4_replicate_dirs = list(v4_dir.glob('replicate_seeds10_19_seed*')) if v4_dir.exists() else []
print(f'historical data check:')
print(f'  v3 n10_wikitext_base dirs found: {len(v3_n10_dirs)}/10')
print(f'  v4 replicate_seeds10_19 dirs found: {len(v4_replicate_dirs)}/10')
print(f'pooled-n≈30 aggregation will use {len(v3_n10_dirs) + len(v4_replicate_dirs)} historical seeds + 10 new = n≈{len(v3_n10_dirs) + len(v4_replicate_dirs) + 10}')

In [ ]:
# 3. Install deps.
!pip install -q "datasets<3"
import sys, torch, numpy as np, datasets
print(f'python: {sys.version.split()[0]} | torch: {torch.__version__} | numpy: {np.__version__} | datasets: {datasets.__version__}')
print(f'cuda available: {torch.cuda.is_available()}; device 0: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

In [ ]:
# 4. Pre-warm wikitext-2 HF cache.
import sys; sys.path.insert(0, '/content/Neuro-AI/src')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
print('warming wikitext-2-raw-v1 cache (parent process, CPU only)...')
splits = load_corpus_splits('wikitext', Path('/content/Neuro-AI'), wikitext_name='wikitext-2-raw-v1')
print(f'  train: {len(splits["train"])} rows; val: {len(splits["validation"])} rows; test: {len(splits["test"])} rows')
del splits; import gc; gc.collect()

In [ ]:
# 5. GPU info.
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 6. SMOKE — confirm runtime + patches sane at the smallest D (D=1024 for cheapness).
import subprocess, sys, os
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
smoke_out = Path('reports/c3_v5_smoke_2026-05-27')
smoke_out.mkdir(parents=True, exist_ok=True)
smoke_log = Path('reports/c3_v5_smoke.log')
cmd = [sys.executable, 'experiments/c3_phase3_exit_criterion.py',
       '--seeds', '0', '--device', 'cuda',
       '--D', '1024',
       '--lr-pull', '0.1', '--lr-push', '0.05',
       '--n-consolidation-events', '100',
       '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
       '--output-dir', str(smoke_out)]
with smoke_log.open('w') as logf:
    rc = subprocess.call(cmd, stdout=logf, stderr=subprocess.STDOUT)
print(f'smoke exit code: {rc}; json: {(smoke_out / "c3_summary.json").exists()}')
print('\n=== smoke log (last 30 lines) ===')
!tail -30 {smoke_log}
if rc != 0:
    raise SystemExit('Smoke failed — abort.')
print('\nSmoke OK.')

In [ ]:
# 7. PARALLEL launch — 4 D values × 10 seeds (30..39) = 40 subprocesses.
import subprocess, os, time, signal, sys
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
PY = sys.executable

D_VALUES = [1024, 2048, 4096, 8192]
SEEDS = list(range(30, 40))

ENTRIES = [(d, seed) for d in D_VALUES for seed in SEEDS]
print(f'launching {len(ENTRIES)} per-seed subprocesses ({len(D_VALUES)} D values × {len(SEEDS)} seeds each)')

log_root = Path('reports/c3_v5_logs')
log_root.mkdir(parents=True, exist_ok=True)

def out_dir_for(d, seed):
    return f'reports/c3_v5_Dsweep_D{d}_seed{seed}_2026-05-27'

def launch(d, seed):
    out_dir = out_dir_for(d, seed)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'D{d}_seed{seed}.log'
    logf = open(log_path, 'w')
    cmd = [PY, 'experiments/c3_phase3_exit_criterion.py',
           '--seeds', str(seed), '--device', 'cuda',
           '--D', str(d),
           '--lr-pull', '0.1', '--lr-push', '0.05',
           '--n-consolidation-events', '1000',
           '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
           '--corpus-source', 'wikitext',
           '--output-dir', out_dir]
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT)
    return proc, logf, out_dir, log_path

def snapshot(remaining, total, t0):
    elapsed = (time.time() - t0) / 60
    n_done = total - len(remaining)
    print(f'  --- snapshot at {elapsed:.1f} min — {n_done}/{total} done, {len(remaining)} running ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    from collections import Counter
    cond_running = Counter()
    for key in remaining:
        cond_running[key.split('_seed')[0]] += 1
    for d in D_VALUES:
        tag = f'D{d}'
        n_run = cond_running.get(tag, 0)
        n_done_tag = len(SEEDS) - n_run
        print(f'    {tag:>10}: {n_done_tag}/{len(SEEDS)} done')

def kill_all(remaining):
    for key, (proc, logf, _, _) in remaining.items():
        try:
            proc.send_signal(signal.SIGKILL); logf.close()
        except Exception:
            pass

# Stagger launches (1.5s × 40 = 60s).
procs = {}
for entry in ENTRIES:
    d, seed = entry
    key = f'D{d}_seed{seed}'
    procs[key] = launch(*entry)
    time.sleep(1.5)
print(f'all {len(procs)} cells launched  ({time.strftime("%H:%M:%S")})')

t0 = time.time()
remaining = dict(procs)
total = len(procs)
failures = []
poll_count = 0
try:
    while remaining:
        done_this_round = []
        for key, (proc, logf, out_dir, log_path) in remaining.items():
            rc = proc.poll()
            if rc is not None:
                logf.close()
                elapsed = (time.time() - t0) / 60
                json_exists = Path(out_dir, 'c3_summary.json').exists()
                ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                print(f'  [{elapsed:5.1f} min] {key:>22}: {ok}  json={json_exists}')
                if rc != 0:
                    failures.append(key)
                    print(f'    --- last 30 lines of {log_path} ---')
                    try:
                        out = subprocess.check_output(['tail', '-30', str(log_path)],
                            stderr=subprocess.DEVNULL).decode()
                        for line in out.splitlines():
                            print(f'    | {line}')
                    except Exception as e:
                        print(f'    | (could not read log: {e})')
                    print('    --- end log ---')
                done_this_round.append(key)
        for key in done_this_round:
            del remaining[key]
        if remaining:
            poll_count += 1
            if poll_count % 3 == 0:
                snapshot(remaining, total, t0)
            time.sleep(30)
except KeyboardInterrupt:
    print('\n!!! Interrupted !!!')
    kill_all(remaining); raise

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
print(f'failures: {len(failures)}/{total}')
if failures:
    print('  failed keys:', failures)
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 7b. EMERGENCY kill.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'c3_phase3_exit_criterion' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL); print(f'  killed {pid}'); killed += 1
        except Exception as e:
            print(f'  err: {e}')
print(f'killed {killed} workers')

In [ ]:
# 8. Copy v5 results + logs to Drive.
import shutil, os
dst_root = '/content/drive/MyDrive/neuro-ai/results/c3_v5_2026-05-27'
os.makedirs(dst_root, exist_ok=True)
D_VALUES = [1024, 2048, 4096, 8192]
SEEDS = list(range(30, 40))
count = 0
for d in D_VALUES:
    for seed in SEEDS:
        src = f'reports/c3_v5_Dsweep_D{d}_seed{seed}_2026-05-27'
        if os.path.isdir(src):
            shutil.copytree(src, f'{dst_root}/Dsweep_D{d}_seed{seed}', dirs_exist_ok=True)
            count += 1
if os.path.isdir('reports/c3_v5_logs'):
    shutil.copytree('reports/c3_v5_logs', f'{dst_root}/colab_logs', dirs_exist_ok=True)
print(f'copied {count} per-seed dirs + logs to {dst_root}')
!ls {dst_root} | head -20

In [ ]:
# 9. AGGREGATION — two tables:
#    (a) D-sweep at n=10 each (new seeds 30..39): shows D-dependence shape.
#    (b) Pooled n≈30 at original op point (D=4096):
#         v3 seeds 0..9 + v4 replicate seeds 10..19 + v5 D=4096 seeds 30..39.
import json, math
from pathlib import Path

D_VALUES = [1024, 2048, 4096, 8192]
SEEDS = list(range(30, 40))
STRATA = ('tight', 'spread', 'borderline')

def wilson(successes, trials, z=1.96):
    if trials == 0: return (0.0, 0.0, 0.0)
    p = successes / trials; n = trials
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = (z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / denom
    return (p, max(0.0, center - half), min(1.0, center + half))

def read_rows_from_dir(parent_dir, glob_pattern):
    """Walk parent_dir for subdirs matching glob_pattern; read each c3_summary.json's per_cell_rows."""
    p = Path(parent_dir)
    if not p.exists():
        return [], 0
    rows = []
    seed_count = 0
    for subdir in sorted(p.glob(glob_pattern)):
        json_path = subdir / 'c3_summary.json'
        if not json_path.exists():
            continue
        d = json.loads(json_path.read_text())
        rows.extend(d['per_cell_rows'])
        seed_count += 1
    return rows, seed_count

def aggregate_rows(rows):
    """Pool per_cell_rows into per (mode, condition, stratum) Wilson-CI table."""
    aggregated = {}
    modes = sorted({r['theta_prime_mode'] for r in rows})
    for mode in modes:
        aggregated[mode] = {}
        for is_control in (False, True):
            key = 'shuffled_control' if is_control else 'standard'
            aggregated[mode][key] = {}
            for stratum in STRATA:
                tot_s = 0; tot_t = 0
                for r in rows:
                    if r['theta_prime_mode'] != mode or r['is_control'] != is_control: continue
                    cell = r['per_stratum'][stratum]
                    tot_s += int(cell['successes']); tot_t += int(cell['trials'])
                mean_v, lo, hi = wilson(tot_s, tot_t)
                aggregated[mode][key][stratum] = {'successes': tot_s, 'trials': tot_t,
                    'recall_at_k': mean_v, 'wilson_lower': lo, 'wilson_upper': hi}
        aggregated[mode]['delta_standard_minus_control'] = {}
        for stratum in STRATA:
            s = aggregated[mode]['standard'][stratum]; c = aggregated[mode]['shuffled_control'][stratum]
            aggregated[mode]['delta_standard_minus_control'][stratum] = {
                'delta_recall_at_k': s['recall_at_k'] - c['recall_at_k'],
                'standard_trials': s['trials'], 'control_trials': c['trials'],
                'ci_disjoint_standard_beats_control': s['wilson_lower'] > c['wilson_upper']}
    return aggregated, modes

def print_aggregated(label, agg, modes, n_seeds):
    print(f'\n--- {label} (n_seeds={n_seeds}) ---')
    for mode in modes:
        for stratum in STRATA:
            s = agg[mode]['standard'][stratum]; c = agg[mode]['shuffled_control'][stratum]
            dl = agg[mode]['delta_standard_minus_control'][stratum]
            if s['trials'] == 0 and c['trials'] == 0: continue
            std_str  = f'{s["recall_at_k"]:.3f} [{s["wilson_lower"]:.3f},{s["wilson_upper"]:.3f}]'
            ctrl_str = f'{c["recall_at_k"]:.3f} [{c["wilson_lower"]:.3f},{c["wilson_upper"]:.3f}]'
            disj = 'YES' if dl['ci_disjoint_standard_beats_control'] else 'no'
            print(f'  {mode:>11} {stratum:>11}  std {std_str:>22}  ctrl {ctrl_str:>22}  '
                  f'Δ {dl["delta_recall_at_k"]:>+7.3f}  disjoint? {disj}  n_std={s["trials"]:>5}  n_ctrl={c["trials"]:>5}')

import os
merged_root = '/content/drive/MyDrive/neuro-ai/results/c3_v5_2026-05-27/_merged'
os.makedirs(merged_root, exist_ok=True)

# === (a) D-sweep table ===
print('=' * 100)
print('(a) D-SWEEP — fresh seeds 30..39 at each D')
print('=' * 100)
dsweep_results = {}
for d in D_VALUES:
    rows, n = read_rows_from_dir('reports', f'c3_v5_Dsweep_D{d}_seed*_2026-05-27')
    if n == 0:
        print(f'\n--- D={d} : ALL MISSING ---')
        continue
    agg, modes = aggregate_rows(rows)
    dsweep_results[d] = (agg, n)
    print_aggregated(f'D={d}', agg, modes, n)
    with open(f'{merged_root}/Dsweep_D{d}.json', 'w') as f:
        json.dump({'D': d, 'n_seeds': n, 'aggregated': agg}, f, indent=2)

# === Compact D-sweep curve ===
print('\n--- D-sweep compact (default/spread Δ, the primary v3 disjoint metric) ---')
print(f'{"D":>5} {"std R@K":>22} {"ctrl R@K":>22} {"Δ":>8} {"disjoint?":>9}')
for d in D_VALUES:
    if d not in dsweep_results: continue
    agg, n = dsweep_results[d]
    if 'default' not in agg: continue
    s = agg['default']['standard']['spread']; c = agg['default']['shuffled_control']['spread']
    dl = agg['default']['delta_standard_minus_control']['spread']
    std_str  = f'{s["recall_at_k"]:.3f} [{s["wilson_lower"]:.3f},{s["wilson_upper"]:.3f}]'
    ctrl_str = f'{c["recall_at_k"]:.3f} [{c["wilson_lower"]:.3f},{c["wilson_upper"]:.3f}]'
    disj = 'YES' if dl['ci_disjoint_standard_beats_control'] else 'no'
    print(f'{d:>5} {std_str:>22} {ctrl_str:>22} {dl["delta_recall_at_k"]:>+8.3f} {disj:>9}')

# === (b) Pooled n≈30 at D=4096 (original operating point) ===
print('\n' + '=' * 100)
print('(b) POOLED LARGER-N at D=4096 (original v3 operating point)')
print('=' * 100)
v3_rows, v3_n = read_rows_from_dir(
    '/content/drive/MyDrive/neuro-ai/results/c3_followup_v3_2026-05-27',
    'n10_wikitext_base_seed*')
v4_rows, v4_n = read_rows_from_dir(
    '/content/drive/MyDrive/neuro-ai/results/c3_robustness_v4_2026-05-27',
    'replicate_seeds10_19_seed*')
v5_rows, v5_n = read_rows_from_dir('reports', 'c3_v5_Dsweep_D4096_seed*_2026-05-27')

print(f'  v3 (seeds 0..9, from Drive):    {v3_n} seeds, {len(v3_rows)} per_cell rows')
print(f'  v4 (seeds 10..19, from Drive):  {v4_n} seeds, {len(v4_rows)} per_cell rows')
print(f'  v5 (seeds 30..39, new):         {v5_n} seeds, {len(v5_rows)} per_cell rows')

all_rows = v3_rows + v4_rows + v5_rows
total_n = v3_n + v4_n + v5_n
if all_rows:
    pooled_agg, pooled_modes = aggregate_rows(all_rows)
    print_aggregated(f'POOLED (v3 + v4 replicate + v5 D=4096)', pooled_agg, pooled_modes, total_n)
    with open(f'{merged_root}/pooled_D4096_n{total_n}.json', 'w') as f:
        json.dump({'D': 4096, 'n_seeds_total': total_n,
                   'v3_seeds': v3_n, 'v4_replicate_seeds': v4_n, 'v5_new_seeds': v5_n,
                   'aggregated': pooled_agg}, f, indent=2)
else:
    print('  no rows available — aggregation skipped.')

# === Verdict ===
print('\n' + '=' * 100)
print('VERDICT GUIDE')
print('=' * 100)
if all_rows and 'default' in pooled_agg:
    pooled_dl = pooled_agg['default']['delta_standard_minus_control']['spread']
    pooled_disjoint = pooled_dl['ci_disjoint_standard_beats_control']
    print(f'  Pooled n={total_n} at D=4096 default/spread: Δ={pooled_dl["delta_recall_at_k"]:+.3f}  disjoint={pooled_disjoint}')
# D=2048 v5 direction
if 2048 in dsweep_results and 'default' in dsweep_results[2048][0]:
    d2048_delta = dsweep_results[2048][0]['default']['delta_standard_minus_control']['spread']['delta_recall_at_k']
    print(f'  D=2048 default/spread Δ (new seeds): {d2048_delta:+.3f} — direction {"REVERSED" if d2048_delta < 0 else "positive"}')
print()
print('Decision tree:')
print('  Pooled disjoint AND D=2048 still reverses → narrow op-point claim; write Report 112 as sensitivity study; pivot to Path γ.')
print('  Pooled disjoint AND D=2048 NOT reversed   → effect real but D=2048 v4 reversal was seed-specific; consider larger-n D-sweep.')
print('  Pooled NOT disjoint                       → v3 disjoint was lucky; signal too weak; clean Path γ pivot, no graduation.')